# 07장 보안 실습 — 계정·권한·GTFOBins 검토


## Goal

UID 0·특수 비트·쓰기 권한을 정상 기준선과 비교하고 GTFOBins 관련성·설정·실행 근거를 분리합니다.

[교안과 분석 질문](../../07-secure-scripting/07-3-account-permission-review.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-07-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'passwd.sample': 'root:x:0:0:root:/root:/bin/bash\nanalyst:x:1000:1000:Analyst:/home/analyst:/bin/bash\ncollector:x:995:995:Collector:/var/lib/collector:/usr/sbin/nologin\nlegacy-admin:x:0:0:Legacy:/var/lib/legacy:/usr/sbin/nologin\n', 'permissions.psv': 'path|owner|group|mode|purpose|approval\n/usr/bin/passwd|root|root|4755|password-management|baseline\n/opt/collector/bin/report|root|collector|0775|service-executable|review\n/var/tmp/course-cache|root|root|1777|shared-temp|baseline\n', 'tool-review.psv': 'case_id|tool_role|reference_listed|context|business_need|approval|scope_fit|telemetry\nR01|text-filter|yes|unprivileged|documented|approved|aligned|present\nR02|report-helper|yes|sudo|unknown|unknown|unknown|not_collected\nR03|custom-helper|no|service|documented|unknown|review|not_collected\nR04|network-helper|yes|capabilities|documented|approved|aligned|present\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: UID 0 계정 2개, 기준선 SUID 1개, 검토 항목 1개; 별도 카드 aligned=2/review=1/unknown=1, 악용 입증 아님

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. UID 0 계정과 로그인 셸 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F: '$3==0 {print $1 "|" $7}' "$COURSE_DATA/passwd.sample" > "$COURSE_OUT/uid0.psv"
test "$(wc -l < "$COURSE_OUT/uid0.psv")" -eq 2
grep -Fx 'legacy-admin|/usr/sbin/nologin' "$COURSE_OUT/uid0.psv"


### 2. 비트와 승인 기준선 함께 읽기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $4 ~ /^[4567]/ {print $1 "|" $4 "|" $6}' \
 "$COURSE_DATA/permissions.psv" > "$COURSE_OUT/suid-review.psv"
grep -Fx '/usr/bin/passwd|4755|baseline' "$COURSE_OUT/suid-review.psv"


### 3. 검토 대상과 취약점 확정 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $6=="review" {print $1 "|" $3 "|" $4}' \
 "$COURSE_DATA/permissions.psv" > "$COURSE_OUT/review.psv"
grep -Fx '/opt/collector/bin/report|collector|0775' "$COURSE_OUT/review.psv"
printf 'review_items=1 exploitation_proven=no\n'


### 4. GTFOBins 검토 카드의 형식과 허용값 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
# Synthetic assessment summaries, not live sudo policy or a binary catalogue.
awk -F '|' '
 NR==1 {
   if ($0!="case_id|tool_role|reference_listed|context|business_need|approval|scope_fit|telemetry") bad=1
   next
 }
 NF!=8 || $1!~/^R[0-9][0-9]$/ || seen[$1]++ || $2=="" ||
 $3!~/^(yes|no)$/ || $4!~/^(unprivileged|sudo|suid|capabilities|service)$/ ||
 $5!~/^(documented|unknown)$/ || $6!~/^(approved|unknown)$/ ||
 $7!~/^(aligned|review|unknown)$/ || $8!~/^(present|not_collected)$/ {bad=1}
 END {if (NR<2 || bad) exit 2}
' "$COURSE_DATA/tool-review.psv"
printf 'review_schema=valid\n'


### 5. 설정 검토와 실행 자료를 서로 다른 축으로 집계


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 {print $1 "|" $3 "|" $7 "|" $8}' \
 "$COURSE_DATA/tool-review.psv" > "$COURSE_OUT/tool-review-results.psv"
awk -F '|' 'NR>1 {scope[$7]++; telemetry[$8]++}
 END {
   printf "aligned=%d review=%d unknown=%d\n", scope["aligned"],scope["review"],scope["unknown"]
   printf "telemetry_present=%d telemetry_not_collected=%d\n",telemetry["present"],telemetry["not_collected"]
 }' "$COURSE_DATA/tool-review.psv" > "$COURSE_OUT/tool-review-counts.txt"
grep -Fx 'aligned=2 review=1 unknown=1' "$COURSE_OUT/tool-review-counts.txt"
grep -Fx 'telemetry_present=2 telemetry_not_collected=2' "$COURSE_OUT/tool-review-counts.txt"
grep -Fx 'R02|yes|unknown|not_collected' "$COURSE_OUT/tool-review-results.psv"
grep -Fx 'R03|no|review|not_collected' "$COURSE_OUT/tool-review-results.psv"
printf 'catalogue_membership_is_not_a_verdict=yes\n'


## GTFOBins 연결

[07-4 전체 해설](../../07-secure-scripting/07-4-gtfobins-review.md)을 읽습니다. tool-review.psv는 가상 검토 카드이며 실제 목록·sudo 정책·사건 증거가 아닙니다. 설정 상태 aligned=2/review=1/unknown=1과 실행 자료 present=2/not_collected=2는 별도 축입니다. R01의 등재만으로 취약, R03의 미등재만으로 안전이라고 결론 내리지 않습니다.


## Red Team ↔ Blue Team 사례 분석

### 사례: UID 0 계정과 쓰기 가능한 업무 파일

**Red Team 질문:** 식별된 계정·권한 예외가 업무 범위를 넘어서는 통제 가능성을 만드는가? UID 0 계정 둘과 그룹 쓰기 가능 파일 하나는 서로 다른 관찰입니다. 두 사실을 연결하는 실행·접근 조건 없이 하나의 권한 상승 경로로 단정하지 않습니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 계정·위임·파일 변경 경계의 과도한 권한 검토 |
| Command / Observation | awk로 UID 0과 권한 요약 읽기, 승인 VM에서는 범위를 정한 find/getcap 조회 |
| System Change / Artifact | 계정·그룹·비트·ACL·Capability·정책의 변경 가능 흔적. 현재 값은 변경 주체를 직접 보여주지 않음 |
| Log prerequisite | 계정 관리·Audit·정책 변경·패키지/배포 기록. sudo 로그는 모든 SUID 실행 기록이 아님 |
| Blue Team Investigation | legacy-admin의 생성·승인·사용 이력, collector 파일의 실행 주체와 수정 권한 비교 |
| Detection | 정상 기준선 밖의 권한 변경과 실제 사용을 별도 경보·조사 상태로 관리 |
| Mitigation | 예외 권한 재검토·업무와 배포 권한 분리·정책 변경 승인. 원본 보존 후 승인 절차로 수정 |

**반례와 해설:** passwd의 SUID는 정상 배포 기준선일 수 있습니다. nologin 설정은 특정 셸 로그인 경로를 제한하지만 UID 0 권한을 없애는 것이 아닙니다. 자동 보고서는 `review`를 `exploited`로 바꾸지 않습니다.

**제출 과제:** 계정 예외와 파일 예외를 별도 항목으로 작성하고, 각각 성립 조건·부족한 자료·정상 반례·완화를 적습니다. 다음 절의 [GTFOBins 검토](../../07-secure-scripting/07-4-gtfobins-review.md)에서 정상 도구의 기능과 실행 권한 문맥을 비교합니다.


## Red Team ↔ Blue Team 사례 분석

### 사례: 업무용 도구 위임의 범위가 문서화되지 않았다

**Red Team 질문:** “파일 보고서 작성이라는 업무에 필요한 권한과 도구가 수행할 수 있는 기능 사이에 차이가 있는가?” 목적은 경계의 과도한 위임 가능성을 식별하는 것입니다. 프로그램 등재만으로 성립 조건이나 영향 범위를 확정하지 않습니다.

| 연결 단계 | 확인·기록할 내용 |
|---|---|
| Technique / Goal | 정상 도구 기능이 의도한 작업 범위를 넘을 가능성이라는 가설 |
| Command / Technique | 제공 권한·업무 요약표를 awk로 읽고, 실제 진단에서는 승인된 설정 사본 검토 |
| Security Meaning | 위임한 업무와 허용 기능이 다르면 최소 권한 검토 필요 |
| System Change / Artifact | 정책 변경 흔적과 도구 실행 흔적은 별개. 파일 읽기는 내용 변경을 남기지 않을 수도 있음 |
| Log / 수집 전제 | sudo 이벤트·승인 이력, 사전 규칙이 있는 Audit/EDR 실행·파일 접근 기록. auth.log가 모든 파일 읽기를 기록하지 않음 |
| Blue Team Investigation | 실행 주체·대상 사용자·인수·파일 범위·시각을 승인 업무와 비교 |
| Detection | 승인 범위와 실행 기록의 차이를 조사 후보로 분류. 도구 이름만으로 경보하지 않음 |
| Mitigation | 업무별 최소 위임, 경로·데이터 접근 통제, 변경 관리, 필요한 감사와 보존을 설계 |

**정상 반례:** 승인된 관리자가 허용된 보고서를 읽었습니다. 이 경우 높은 권한의 실행 기록이 있어도 업무 범위 일치 여부를 먼저 확인해야 합니다.

**자료 부족 사례:** 허용 정책만 있고 실행 기록이 수집되지 않았습니다. “위임 범위 확인 필요, 사용 여부 미확인”까지 보고할 수 있으며 “침해 없음”이나 “권한 상승 성공”은 모두 근거를 넘습니다.

관련 분류는 [Sudo and Sudo Caching — T1548.003](https://attack.mitre.org/techniques/T1548/003/)과 [Setuid and Setgid — T1548.001](https://attack.mitre.org/techniques/T1548/001/)입니다. 정상 정책 검토에 공격 발생 판정을 붙이는 ID가 아닙니다. 실제 행위 문맥과 근거가 맞을 때만 분석 보고서에 연결합니다.

### 탐지 설계 연습

탐지 요구사항을 “GTFOBins 이름 발견 시 경보”가 아니라 “업무 승인 범위와 권한 있는 실행의 불일치 검토”로 작성합니다. 입력에는 호스트·사건 시각·주체·대상 권한·실행 경로·정책 버전·승인 범위가 필요합니다. 사전에 기록되지 않은 필드는 unknown으로 남기고 경보의 확신도를 높이는 근거로 쓰지 않습니다.

평가 데이터는 최소 세 종류를 준비합니다: 승인 업무와 일치, 승인 범위와 차이, 로그 미수집. 각각 정상 검토·추가 검토·판단 보류로 구분되는지 확인합니다. 정책 변경 뒤에는 정상 업무도 재검증하며, 탐지를 통과했다고 최소 권한 설정이 보장되는 것은 아닙니다.


## 역할별 분석 기록

같은 실행 결과로 아래 항목을 작성하고 상대 관점에서 검토합니다. 자동 테스트는 계산과 원본 보존만 확인하며 이 서술 과제는 강사 또는 동료 검토 대상입니다.

| 항목 | 학생 작성 |
|---|---|
| Red Team 목적·필요 조건 | 관찰에서 도출한 질문과 전제 |
| 실제 관찰 | 파일·행·이벤트 ID와 출력 |
| Artifact·로깅 전제 | 확보한 자료와 필요한 기록 기능 |
| Blue Team 조사 | 정상 반례·추가 근거·수집 한계 |
| 탐지·완화 | 필요한 필드·오탐 사례·확인된 원인에 맞는 조치 |


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
